In [1]:

import numpy as np

from scripts.data_loader import load_dataframe
from scripts.utils import NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt('../data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

print(f"Loaded df of size {df.shape}")

Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5
Loaded df of size (33537, 4)


In [2]:
from scripts.utils import dataset_tp_rp_split
from scripts.localization_model import LocalizationModel
from time import perf_counter

data = df
df_tp, df_rp = dataset_tp_rp_split(data, 0.3, 42)

loc_model = LocalizationModel()

print(f"Fitting the Localization model")
start = perf_counter()
loc_model.fit(df_rp)
end = perf_counter()
training_time = end - start

print(f"Prediciton positions")
start = perf_counter()
df_tp[['est_lat', 'est_lng']] = loc_model.predict(df_tp)
end = perf_counter()
inference_time = end - start

Fitting the Localization model
Prediciton positions


In [3]:
from scripts.utils import haversine_distance

df_tp["error_m"] = haversine_distance(
    df_tp["lat"].values,
    df_tp["lng"].values,
    df_tp["est_lat"].values,
    df_tp["est_lng"].values,
)
print(f"""
RESULTS
mean:\t {df_tp["error_m"].mean():.2f} m
std:\t {df_tp["error_m"].std():.2f} m
median:\t {df_tp["error_m"].median():.2f} m
max:\t {df_tp["error_m"].max():.2f} m
min:\t {df_tp["error_m"].min():.2f} m

TIMING
training:\t {training_time:.2f} s
inference:\t {inference_time:.2f} s
""")



RESULTS
mean:	 2.97 m
std:	 4.93 m
median:	 0.66 m
max:	 82.36 m
min:	 0.00 m

TIMING
training:	 58.58 s
inference:	 117.79 s

